# Zadanie 2: RAG na własnym korpusie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/02-rag/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-02`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 3.

Części:
1. Korpus z metadanymi, chunking i Chroma z filtrem.
2. Wyszukiwanie: wektory, BM25, hybryda, reranking i porównanie na zestawie testowym.
3. Pipeline RAG i ocena odpowiedzi przez LLM.

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

In [2]:
import os

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-4o-mini", "classic": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

czat: proxy (copilot/gpt-5-mini, klasyczny: copilot/gpt-5-mini) | embeddingi: ollama (embeddinggemma)


## Część 1. Korpus, chunking, Chroma

1. Korpus: **co najmniej 10 dokumentów** (każdy min. 400 znaków) z wybranej dziedziny, np. regulamin studiów, instrukcje sprzętu, przepisy kulinarne, dokumentacja biblioteki. Można też użyć korpusu z zajęć (`data/`) i dopisać do niego **5 własnych dokumentów**. Każdy dokument ma co najmniej jedno pole metadanych, po którym ma sens filtrować (kategoria, rok, wersja, język).
2. Chunking `RecursiveCharacterTextSplitter` z wybranym `chunk_size` i `chunk_overlap`; wypisanie liczby chunków i jednego przykładowego chunku z metadanymi.
3. Kolekcja Chroma i **jedno zapytanie**, dla którego wynik z filtrem `where` różni się od wyniku bez filtra. Wypisanie obu list.

In [ ]:
from pathlib import Path

def load_docs(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.md")):
        title, meta, _, *body = path.read_text(encoding="utf-8").split("\n")
        meta = dict(kv.split(": ") for kv in meta.split(" | "))
        docs.append({"id": path.stem, "title": title.lstrip("# "), "dzial": meta["dział"],
                     "rok": int(meta["rok"]), "text": "\n".join(body).strip()})
    return docs

docs = load_docs("data")
print(f"dokumentów: {len(docs)}, znaków razem: {sum(len(d['text']) for d in docs)}")
for d in docs[:5]:
    print(f"  {d['id']:<24} {d['dzial']:<8} {d['rok']}  {len(d['text'])} znaków")

dokumentów: 20, znaków razem: 13451
  benefity-2026            HR       2026  492 znaków
  benefity                 HR       2025  731 znaków
  biuro                    Administracja 2025  730 znaków
  delegacje                Finanse  2026  784 znaków
  hasla-i-klucze           IT       2026  768 znaków


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def make_chunks(docs, size=300, overlap=50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = []
    for d in docs:
        for i, part in enumerate(splitter.split_text(d["text"])):
            chunks.append({
				"id": f"{d['id']}#{i}", 
                "text": f"{d['title']}\n{part}",
                "meta": {
                    "doc": d["id"],
                    "dzial": d["dzial"],
                    "rok": d["rok"]
                }
            })
    return chunks

chunks = make_chunks(docs, size=200, overlap=50)
print(len(chunks), "chunków")
print(chunks[70]["id"], chunks[70]["meta"])
print(chunks[70]["text"])

99 chunków
rowerem-do-biura#2 {'doc': 'rowerem-do-biura', 'dzial': 'Administracja', 'rok': 2026}
Dojazdy rowerem i prysznice
na korytarzu ani przypinać do ogrodzenia. Usterki stojaków i wyposażenia należy zgłaszać administracji przez formularz Helpdesk.


In [ ]:
import chromadb

__cache = {}
def embed(texts):
    """Embeddingi dla listy tekstów (z cache)."""
    todo = [t for t in texts if t not in __cache]
    if todo:
        for t, d in zip(todo, embed_client.embeddings.create(model=EMBED, input=todo).data):
            __cache[t] = d.embedding
    return [__cache[t] for t in texts]

def build_collection(chunks, name="firma"):
    db = chromadb.Client()
    if name in [c.name for c in db.list_collections()]:
        db.delete_collection(name)
    col = db.create_collection(name, metadata={"hnsw:space": "cosine"})
    col.add(
        ids=[c["id"] for c in chunks], 
        documents=[c["text"] for c in chunks],
		metadatas=[c["meta"] for c in chunks], 
        embeddings=embed([c["text"] for c in chunks])
    )
    return col

collection = build_collection(chunks)
print("w kolekcji:", collection.count())

w kolekcji: 99


In [27]:
def vector_search(query, k=3, where=None):
    res = collection.query(query_embeddings=embed([query]), n_results=k, where=where)
    return list(zip(res["ids"][0], res["distances"][0], res["metadatas"][0]))

q = "Jaki pakiet prywatnej opieki zdrowotnej oferuje firma?"
for where in [None, {"rok": {"$gte": 2026}}]:
    print("where =", where)
    for cid, dist, meta in vector_search(q, where=where):
        print(f"   {dist:.3f}  {cid:<22} {meta['dzial']} {meta['rok']}")

where = None
   0.384  benefity#1             HR 2025
   0.394  benefity-2026#1        HR 2026
   0.525  benefity-2026#0        HR 2026
where = {'rok': {'$gte': 2026}}
   0.394  benefity-2026#1        HR 2026
   0.525  benefity-2026#0        HR 2026
   0.532  lumen-sync-cennik#1    Produkt 2026


In [33]:
for chunk in filter(lambda c: c["id"] in ["benefity#1", "benefity-2026#1"], chunks):
	print(chunk)

{'id': 'benefity-2026#1', 'text': 'Benefity pracownicze (wersja 2026)\nPodstawowy pakiet prywatnej opieki medycznej jest finansowany w 50% przez firmę, a pozostałą część opłaca pracownik. Pakiety rozszerzone i wizyty stomatologiczne pozostają dodatkowo płatne. Dopłata', 'meta': {'doc': 'benefity-2026', 'dzial': 'HR', 'rok': 2026}}
{'id': 'benefity#1', 'text': 'Benefity pracownicze\nPrywatna opieka medyczna obejmuje pakiet podstawowy finansowany w całości przez firmę. Rozszerzenie pakietu o stomatologię i rehabilitację jest płatne przez pracownika.', 'meta': {'doc': 'benefity', 'dzial': 'HR', 'rok': 2025}}


## Część 2. Porównanie metod wyszukiwania

1. Zestaw testowy: **co najmniej 10 pytań** z oznaczonym dokumentem, w którym jest odpowiedź. Co najmniej 3 pytania to parafrazy (inne słowa niż w dokumencie), co najmniej 2 zawierają nazwę własną, liczbę albo kod.
2. Cztery metody: wektory, BM25, hybryda (RRF), hybryda + reranking cross-encoderem.
3. Tabela: `hit@1`, `hit@3`, `MRR` dla każdej metody.
4. Ta sama tabela (tylko wektory) dla **dwóch innych** ustawień `chunk_size`.
5. W komórce markdown (3–5 zdań): która metoda wypadła najlepiej, na którym pytaniu metody się różniły i dlaczego.

In [ ]:
EVAL = [
    (
        "Jaki procent kosztu podstawowego pakietu medycznego pokrywa firma po zmianach w 2026 roku?",
        "benefity-2026",
        "Firma finansuje 50% podstawowego pakietu prywatnej opieki medycznej, a pozostałą część opłaca pracownik.",
    ),
    (
        "Jak według zasad z 2025 roku finansowany był podstawowy pakiet prywatnej opieki medycznej?",
        "benefity",
        "Podstawowy pakiet prywatnej opieki medycznej był finansowany w całości przez firmę.",
    ),
    (
        "Ile dni przed szkoleniem trzeba wysłać wniosek w Kadrus?",
        "szkolenia-2026",
        "Wniosek należy przesłać co najmniej dziesięć dni przed rozpoczęciem szkolenia.",
    ),
    (
        "Gdzie znajduje się zamykana stojakownia dla rowerów?",
        "rowerem-do-biura",
        "Stojakownia znajduje się przy wejściu do biura od strony ulicy Jasnej.",
    ),
    (
        "Jak pracownik uzyskuje dostęp do pryszniców i stojakowni?",
        "rowerem-do-biura",
        "Dostęp do zamykanej stojakowni wydaje recepcja po okazaniu identyfikatora, a prysznice są dostępne w szatni.",
    ),
    (
        "Po jakim czasie od zatrudnienia można aplikować w rekrutacji wewnętrznej?",
        "rekrutacja-wewnetrzna",
        "Można aplikować po zakończeniu trzymiesięcznego okresu próbnego.",
    ),
    (
        "Czy obecny przełożony może zablokować przesłanie formularza do rekrutacji wewnętrznej?",
        "rekrutacja-wewnetrzna",
        "Nie. Zgoda obecnego przełożonego nie jest wymagana do przesłania formularza.",
    ),
    (
        "Jaki jest maksymalny czas cotygodniowego spotkania statusowego?",
        "spotkania-zespolowe",
        "Cotygodniowe spotkanie statusowe trwa maksymalnie trzydzieści minut.",
    ),
    (
        "Gdzie zapisuje się decyzje wymagające udziału większej grupy osób?",
        "spotkania-zespolowe",
        "Takie decyzje zapisuje się w przestrzeni projektowej.",
    ),
    (
        "Ile dni pracy zdalnej tygodniowo dopuszcza polityka Lumen Sync z 2026 roku?",
        "praca-zdalna-2026",
        "Pracownik może pracować zdalnie do trzech dni w tygodniu.",
    ),
]

In [ ]:
# TODO: BM25, RRF, reranking, funkcja evaluate i tabela wyników

In [ ]:
# TODO: wpływ chunk_size

*Wnioski:*

## Część 3. RAG i ocena odpowiedzi

1. Funkcja `rag(question)`: wyszukiwanie (dowolna metoda z części 2) → prompt z kontekstem → odpowiedź z identyfikatorami źródeł.
2. Odpowiedzi na **co najmniej 5 pytań** z zestawu testowego oraz **1 pytanie, na które korpus nie zawiera odpowiedzi** (oczekiwane „Nie wiem”).
3. LLM jako sędzia: wierność (faithfulness) i poprawność każdej odpowiedzi, z jednozdaniowym uzasadnieniem, oraz średnie.
4. W komórce markdown (2–4 zdania): czy któraś ocena sędziego jest błędna; jeśli wszystkie są trafne, jaki przypadek mógłby go zmylić.

In [ ]:
# TODO: rag() i odpowiedzi

In [ ]:
# TODO: ocena przez LLM

*Komentarz do ocen sędziego:*